In [11]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
import itertools
import matplotlib.pyplot as plt

In [9]:
prices = pd.read_csv(
    "./dataset/cleaned_data.csv", parse_dates=["Date"], index_col="Date"
)

print("Price matrix:", prices.shape)
display(prices.head())

Price matrix: (3905, 432)


,GNRC,CHTR,MTCH,NDAQ,WFC,WM,WELL,WMB,MU,NDSN,...,EXPE,EXPD,EXC,EW,EVRG,ETR,ETN,ESS,ES,IT
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,12.84,35.0,5.856653,6.746667,27.320000,34.160000,44.040001,17.616472,10.85,31.530001,...,51.639999,35.080002,34.864479,7.289167,21.840000,41.075001,32.160000,82.620003,25.770000,18.700001
2010-01-05,12.84,35.0,5.985151,6.766667,28.070000,34.009998,44.660000,17.836576,11.17,31.490000,...,51.860001,35.320000,34.293865,7.341667,21.549999,40.419998,31.969999,83.120003,25.680000,18.730000
2010-01-06,12.84,35.0,5.936608,6.763333,28.110001,34.000000,44.439999,18.415367,11.22,31.139999,...,49.200001,34.500000,34.500713,7.422500,21.670000,40.625000,31.830000,83.699997,26.010000,18.860001
2010-01-07,12.84,35.0,5.965163,6.673333,29.129999,34.080002,44.529999,18.284937,10.84,31.514999,...,48.939999,34.290001,34.614838,7.490000,21.530001,40.139999,32.299999,84.570000,25.969999,20.440001
2010-01-08,12.84,33.5,6.002284,6.743333,28.860001,34.240002,44.070000,18.431671,11.10,31.875000,...,48.480000,34.650002,34.450787,7.456667,21.719999,39.755001,33.025002,83.699997,26.040001,20.660000


In [15]:
def run_coint(pair):
    a, b = pair
    s1 = prices[a].dropna()
    s2 = prices[b].dropna()
    _, pvalue, _ = coint(s1, s2)
    return (pair, pvalue)

In [19]:
from pathos.multiprocessing import ProcessingPool as Pool
from statsmodels.tsa.stattools import coint
import itertools
from tqdm import tqdm
import pandas as pd

tickers = prices.columns.tolist()
pairs = list(itertools.combinations(tickers, 2))


def run_coint(pair):
    a, b = pair
    s1 = prices[a].dropna()
    s2 = prices[b].dropna()
    _, pvalue, _ = coint(s1, s2)
    return (pair, pvalue)


# Use tqdm.imap to show progress
pool = Pool()
results = []
for res in tqdm(
    pool.imap(run_coint, pairs), total=len(pairs), desc="Cointegration tests"
):
    results.append(res)

pool.close()
pool.join()

coint_results = pd.DataFrame(results, columns=["pair", "pvalue"]).sort_values("pvalue")
coint_results.head()

Cointegration tests:   0%|          | 0/93096 [03:54<?, ?it/s]


KeyboardInterrupt: 